# JSONB with SQLAlchemy: playground

Interactive version of the scripts in this folder (`01`–`04`). Same `Event` model, same five
sample rows, but every query returns a pandas DataFrame so you can look at, filter and reshape
results in place.

**Prerequisites**: the Postgres container is up (`docker compose up -d` from the project root)
and the kernel is this project's `.venv` (`uv sync` created it; pick it in the kernel selector).

The notebook uses top-level `await`, which Jupyter supports out of the box.

In [94]:
# Locate the project root (works whether the kernel's cwd is the project root or this folder),
# put src/ on the path and load .env so DATABASE_URL etc. are set.
import pathlib
import sys

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

import pandas as pd
from sqlalchemy import (
    Integer,
    Numeric,
    Text,
    column,
    func,
    literal,
    select,
    true,
    update,
)
from sqlalchemy.dialects import postgresql
from sqlalchemy.dialects.postgresql import JSONB, JSONPATH, array

# Local modules
from db import SessionLocal
from examples.jsonb._common import reset_events
from models import Event

pd.set_option("display.max_colwidth", 120)


def sql(stmt) -> None:
    """Print the SQL SQLAlchemy will send, plus bound parameters."""
    c = stmt.compile(dialect=postgresql.dialect())
    print(str(c).strip())
    if c.params:
        print("-- params:", c.params)


async def q(stmt, show=True) -> pd.DataFrame:
    """Execute a statement in the shared session and return the rows as a DataFrame."""
    if show:
        sql(stmt)
    result = await session.execute(stmt)
    return pd.DataFrame(result.mappings().all())


session = SessionLocal()
await reset_events(session)
print("ready")

2026-09-04 20:45:18,946 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 20:45:18,947 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR
2026-09-04 20:45:18,947 INFO sqlalchemy.engine.Engine [cached since 1569s ago] {'table_name': 'events', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-09-04 20:45:18,951 INFO sqlalchemy.engine.Engine COMMIT
2026-09-04 20:45:18,952 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 20:45:18,953 INFO sqlalche

## 0. The data

Five events. `payload` is a `JSONB` column; on the Python side it is a plain dict.

In [ ]:
select_cols = [Event.id, Event.kind, Event.payload]

await q(
    select(*select_cols).order_by(Event.id.desc()),  # Remind me why *, and for dict **
    show=True,
)

SELECT events.id, events.kind, events.payload 
FROM events ORDER BY events.id DESC
2026-09-04 20:41:55,428 INFO sqlalchemy.engine.Engine SELECT events.id, events.kind, events.payload 
FROM events ORDER BY events.id DESC
2026-09-04 20:41:55,428 INFO sqlalchemy.engine.Engine [cached since 63.31s ago] {}


,id,kind,payload
0,120,error,"{'code': 500, 'tags': ['api'], 'message': 'upstream timeout'}"
1,119,purchase,"{'tags': [], 'user': {'id': 3, 'name': 'Chidi', 'country': 'NG'}, 'items': [{'qty': 1, 'sku': 'C-3', 'price': 120}],..."
2,118,purchase,"{'tags': ['promo'], 'user': {'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}, 'items': [{'qty': 2, 'sku': 'A..."
3,117,login,"{'tags': ['new'], 'user': {'id': 2, 'name': 'Bola', 'country': 'GH'}, 'device': 'mobile'}"
4,116,login,"{'tags': ['returning', '2fa'], 'user': {'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}, 'device': 'web'}"


> **Q: why `*select_cols`, and when `**`?**
>
> `select()` takes its columns as separate positional arguments, not as one list:
> `select(Event.id, Event.kind)` works, `select([Event.id, Event.kind])` does not (that was 1.x syntax).
> `*` unpacks a list/tuple into positional arguments, so `select(*select_cols)` is exactly
> `select(Event.id, Event.kind, Event.payload)`.
>
> `**` does the same for keyword arguments, from a dict with string keys:
> `update(Event).values(**{"kind": "x", "payload": {...}})` is `update(Event).values(kind="x", payload={...})`.
> Rule of thumb: `*` for "these go where the function expects positional args", `**` for "these are `name=value` pairs".
> You can mix them: `select(*cols).where(*filters)`, `Event(**row)` (that is how `reset_events` builds the sample rows).

## 1. Reading values out of the document

| SQL | SQLAlchemy |
|---|---|
| `payload -> 'k'` | `Event.payload["k"]` (stays jsonb) |
| `payload ->> 'k'` | `Event.payload["k"].astext` (text) |
| `payload #>> '{a,b}'` | `Event.payload[("a", "b")].astext` |
| `(payload ->> 'k')::int` | `Event.payload["k"].as_integer()` |

In [ ]:
# Need to memorize these patterns
select_cols = [
    Event.id,
    Event.payload["user"].label("user_obj"),  # ->   nested dict
    Event.payload["device"].astext.label("device"),  # ->>  text
    Event.payload[("user", "country")].astext.label("country"),  # #>>  path
    Event.payload["tags"][0].astext.label("first_tag"),  # array index
    Event.payload["amount"].as_float().label("amount"),  # typed cast
]

await q(select(*select_cols).order_by(Event.id))

SELECT events.id, events.payload[%(payload_1)s] AS user_obj, events.payload[%(payload_2)s] AS device, events.payload #>> %(payload_3)s AS country, (events.payload[%(payload_4)s]) ->> %(param_1)s AS first_tag, CAST(events.payload ->> %(payload_5)s AS FLOAT) AS amount 
FROM events ORDER BY events.id
-- params: {'payload_1': 'user', 'payload_2': 'device', 'payload_3': ('user', 'country'), 'payload_4': 'tags', 'param_1': 0, 'payload_5': 'amount'}
2026-09-04 20:43:02,784 INFO sqlalchemy.engine.Engine SELECT events.id, events.payload[%(payload_1)s::TEXT] AS user_obj, events.payload[%(payload_2)s::TEXT] AS device, events.payload #>> %(payload_3)s AS country, (events.payload[%(payload_4)s::TEXT]) ->> %(param_1)s::INT AS first_tag, CAST(events.payload ->> %(payload_5)s::TEXT AS FLOAT) AS amount 
FROM events ORDER BY events.id
2026-09-04 20:43:02,785 INFO sqlalchemy.engine.Engine [generated in 0.00125s] {'payload_1': 'user', 'payload_2': 'device', 'payload_3': '{user, country}', 'payload_4': 'ta

,id,user_obj,device,country,first_tag,amount
0,116,"{'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}",web,NG,returning,NaN
1,117,"{'id': 2, 'name': 'Bola', 'country': 'GH'}",mobile,GH,new,NaN
2,118,"{'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}",NaN,NG,promo,4500.5
3,119,"{'id': 3, 'name': 'Chidi', 'country': 'NG'}",NaN,NG,NaN,120.0
4,120,None,NaN,NaN,api,NaN


In [113]:
await q(select(Event.payload))  # This shows the original payload

SELECT events.payload 
FROM events
2026-09-04 21:14:09,514 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 21:14:09,514 INFO sqlalchemy.engine.Engine SELECT events.payload 
FROM events
2026-09-04 21:14:09,515 INFO sqlalchemy.engine.Engine [cached since 2278s ago] {}


,payload
0,"{'tags': ['returning', '2fa'], 'user': {'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}, 'device': 'web'}"
1,"{'tags': ['new'], 'user': {'id': 2, 'name': 'Bola', 'country': 'GH'}, 'device': 'mobile'}"
2,"{'tags': ['promo'], 'user': {'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}, 'items': [{'qty': 2, 'sku': 'A..."
3,"{'tags': [], 'user': {'id': 3, 'name': 'Chidi', 'country': 'NG'}, 'items': [{'qty': 1, 'sku': 'C-3', 'price': 120}],..."
4,"{'code': 500, 'tags': ['api'], 'message': 'upstream timeout'}"


In [ ]:
await q(select(Event.payload["user"]))  # This shows the user dictionary

SELECT events.payload[%(payload_1)s] AS anon_1 
FROM events
-- params: {'payload_1': 'user'}
2026-09-04 20:43:54,644 INFO sqlalchemy.engine.Engine SELECT events.payload[%(payload_1)s::TEXT] AS anon_1 
FROM events
2026-09-04 20:43:54,645 INFO sqlalchemy.engine.Engine [cached since 1072s ago] {'payload_1': 'user'}


,_no_label
0,"{'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}"
1,"{'id': 2, 'name': 'Bola', 'country': 'GH'}"
2,"{'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}"
3,"{'id': 3, 'name': 'Chidi', 'country': 'NG'}"
4,None


Missing keys come back as SQL `NULL` (pandas shows `None`/`NaN`). That matters for filters, see below.

## 2. Filtering

`contains` (`@>`) is the workhorse and is GIN-indexable. Key-existence operators need a Postgres
text array, so wrap lists in `array([...])`.

In [81]:
await q(
    select(Event.id, Event.kind).where(
        Event.payload.contains({"user": {"country": "NG"}})
    )
)

SELECT events.id, events.kind 
FROM events 
WHERE events.payload @> %(payload_1)s::JSONB
-- params: {'payload_1': {'user': {'country': 'NG'}}}
2026-09-04 20:44:05,235 INFO sqlalchemy.engine.Engine SELECT events.id, events.kind 
FROM events 
WHERE events.payload @> %(payload_1)s::JSONB
2026-09-04 20:44:05,235 INFO sqlalchemy.engine.Engine [cached since 1010s ago] {'payload_1': Jsonb({'user': {'country': 'NG'}})}


,id,kind
0,116,login
1,118,purchase
2,119,purchase


In [82]:
# Return Payloads matching tags pattern
await q(select(Event.payload).where(Event.payload.contains({"tags": ["promo"]})))

SELECT events.payload 
FROM events 
WHERE events.payload @> %(payload_1)s::JSONB
-- params: {'payload_1': {'tags': ['promo']}}
2026-09-04 20:44:06,891 INFO sqlalchemy.engine.Engine SELECT events.payload 
FROM events 
WHERE events.payload @> %(payload_1)s::JSONB
2026-09-04 20:44:06,891 INFO sqlalchemy.engine.Engine [cached since 952.7s ago] {'payload_1': Jsonb({'tags': ['promo']})}


,payload
0,"{'tags': ['promo'], 'user': {'id': 1, 'name': 'Ada', 'plan': 'pro', 'country': 'NG'}, 'items': [{'qty': 2, 'sku': 'A..."


In [37]:
await q(
    select(Event.id, Event.kind).where(Event.payload.has_any(array(["amount", "code"])))
)

SELECT events.id, events.kind 
FROM events 
WHERE events.payload ?| ARRAY[%(param_1)s, %(param_2)s]
-- params: {'param_1': 'amount', 'param_2': 'code'}
2026-09-04 20:28:29,208 INFO sqlalchemy.engine.Engine SELECT events.id, events.kind 
FROM events 
WHERE events.payload ?| ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR]
2026-09-04 20:28:29,208 INFO sqlalchemy.engine.Engine [generated in 0.00058s] {'param_1': 'amount', 'param_2': 'code'}


,id,kind
0,118,purchase
1,119,purchase
2,120,error


In [83]:
# JSONPath: look inside arrays without unnesting
await q(select(Event.id).where(Event.payload.path_exists("$.items[*] ? (@.qty > 1)")))

SELECT events.id 
FROM events 
WHERE events.payload @? %(payload_1)s
-- params: {'payload_1': '$.items[*] ? (@.qty > 1)'}
2026-09-04 20:44:09,850 INFO sqlalchemy.engine.Engine SELECT events.id 
FROM events 
WHERE events.payload @? %(payload_1)s
2026-09-04 20:44:09,850 INFO sqlalchemy.engine.Engine [cached since 936.2s ago] {'payload_1': '$.items[*] ? (@.qty > 1)'}


,id
0,118


**The NULL trap.** `device != 'web'` silently drops every row that has no `device` key.

In [84]:
print(
    "without is_(None):",
    (
        await q(
            select(Event.id).where(Event.payload["device"].astext != "web"), show=False
        )
    )["id"].tolist(),
)
print(
    "with    is_(None):",
    (
        await q(
            select(Event.id).where(
                (Event.payload["device"].astext != "web")
                | (Event.payload["device"].astext.is_(None))
            ),
            show=False,
        )
    )["id"].tolist(),
)

2026-09-04 20:44:12,167 INFO sqlalchemy.engine.Engine SELECT events.id 
FROM events 
WHERE (events.payload ->> %(payload_1)s::TEXT) != %(param_1)s::VARCHAR
2026-09-04 20:44:12,167 INFO sqlalchemy.engine.Engine [cached since 929.4s ago] {'payload_1': 'device', 'param_1': 'web'}
without is_(None): [117]
2026-09-04 20:44:12,171 INFO sqlalchemy.engine.Engine SELECT events.id 
FROM events 
WHERE (events.payload ->> %(payload_1)s::TEXT) != %(param_1)s::VARCHAR OR (events.payload ->> %(payload_2)s::TEXT) IS NULL
2026-09-04 20:44:12,172 INFO sqlalchemy.engine.Engine [cached since 929.4s ago] {'payload_1': 'device', 'param_1': 'web', 'payload_2': 'device'}
with    is_(None): [117, 118, 119, 120]


## 3. Unnesting arrays: `jsonb_array_elements`

One row holding an array becomes one row per element. After that, ordinary `WHERE`, `GROUP BY`
and `JOIN` apply. `.table_valued(column("value", JSONB))` tells SQLAlchemy the element type;
`.join(item, true())` is a cross join that Postgres treats as `LATERAL`.

In [85]:
items_payload = await q(select(Event.payload["items"].label("user_items"), Event.id))
display(items_payload)
print("\tOur dataframe contains items only for two rows.\n")

item = (
    func.jsonb_array_elements(Event.payload["items"])
    .table_valued(column("value", JSONB))
    .render_derived()
)
sku = item.c.value["sku"].astext.label("sku")
qty = item.c.value["qty"].as_integer().label("qty")
price = item.c.value["price"].astext.cast(Numeric(12, 2)).label("price")

items_df = await q(
    select(Event.id, sku, qty, price)
    .join(item, true())
    .where(Event.kind == "purchase")
    .order_by(Event.id, sku)
)
print("\n\tThis operation limits only to records that can be extracted.\n")
display(items_df)

SELECT events.payload[%(payload_1)s] AS user_items, events.id 
FROM events
-- params: {'payload_1': 'items'}
2026-09-04 20:44:14,457 INFO sqlalchemy.engine.Engine SELECT events.payload[%(payload_1)s::TEXT] AS user_items, events.id 
FROM events
2026-09-04 20:44:14,458 INFO sqlalchemy.engine.Engine [cached since 629.1s ago] {'payload_1': 'items'}


,user_items,id
0,None,116
1,None,117
2,"[{'qty': 2, 'sku': 'A-1', 'price': 1000}, {'qty': 1, 'sku': 'B-7', 'price': 2500.5}]",118
3,"[{'qty': 1, 'sku': 'C-3', 'price': 120}]",119
4,None,120


	Our dataframe contains items only for two rows.

SELECT events.id, anon_1.value ->> %(value_1)s AS sku, CAST(anon_1.value ->> %(value_2)s AS INTEGER) AS qty, CAST(anon_1.value ->> %(value_3)s AS NUMERIC(12, 2)) AS price 
FROM events JOIN jsonb_array_elements(events.payload[%(payload_1)s]) AS anon_1(value) ON true 
WHERE events.kind = %(kind_1)s ORDER BY events.id, sku
-- params: {'value_1': 'sku', 'value_2': 'qty', 'value_3': 'price', 'payload_1': 'items', 'kind_1': 'purchase'}
2026-09-04 20:44:14,463 INFO sqlalchemy.engine.Engine SELECT events.id, anon_1.value ->> %(value_1)s::TEXT AS sku, CAST(anon_1.value ->> %(value_2)s::TEXT AS INTEGER) AS qty, CAST(anon_1.value ->> %(value_3)s::TEXT AS NUMERIC(12, 2)) AS price 
FROM events JOIN jsonb_array_elements(events.payload[%(payload_1)s::TEXT]) AS anon_1(value) ON true 
WHERE events.kind = %(kind_1)s::VARCHAR ORDER BY events.id, sku
2026-09-04 20:44:14,463 INFO sqlalchemy.engine.Engine [cached since 919.6s ago] {'value_1': 'sku', 'value_2

,id,sku,qty,price
0,118,A-1,2,1000.00
1,118,B-7,1,2500.50
2,119,C-3,1,120.00


In [86]:
# Aggregate in SQL ...
await q(
    select(sku, func.sum(qty).label("units"), func.sum(qty * price).label("revenue"))
    .select_from(Event)  # needed: no events.* column in SELECT
    .join(item, true())  # How does join work?
    .group_by(sku)
    .order_by(func.sum(qty * price).desc())
)

SELECT anon_1.value ->> %(value_1)s AS sku, sum(CAST(anon_1.value ->> %(value_2)s AS INTEGER)) AS units, sum(CAST((anon_1.value ->> %(value_2)s) AS INTEGER) * CAST((anon_1.value ->> %(value_3)s) AS NUMERIC(12, 2))) AS revenue 
FROM events JOIN jsonb_array_elements(events.payload[%(payload_1)s]) AS anon_1(value) ON true GROUP BY anon_1.value ->> %(value_1)s ORDER BY sum(CAST((anon_1.value ->> %(value_2)s) AS INTEGER) * CAST((anon_1.value ->> %(value_3)s) AS NUMERIC(12, 2))) DESC
-- params: {'value_1': 'sku', 'value_2': 'qty', 'value_3': 'price', 'payload_1': 'items'}
2026-09-04 20:44:17,328 INFO sqlalchemy.engine.Engine SELECT anon_1.value ->> %(value_1)s::TEXT AS sku, sum(CAST(anon_1.value ->> %(value_2)s::TEXT AS INTEGER)) AS units, sum(CAST((anon_1.value ->> %(value_2)s::TEXT) AS INTEGER) * CAST((anon_1.value ->> %(value_3)s::TEXT) AS NUMERIC(12, 2))) AS revenue 
FROM events JOIN jsonb_array_elements(events.payload[%(payload_1)s::TEXT]) AS anon_1(value) ON true GROUP BY anon_1.value 

,sku,units,revenue
0,B-7,1,2500.50
1,A-1,2,2000.00
2,C-3,1,120.00


> **Q: how does `.join(item, true())` work? Joined on what?**
>
> The compiled SQL is:
>
> ```sql
> FROM events JOIN jsonb_array_elements(events.payload -> 'items') AS anon_1(value) ON true
> ```
>
> Two things are going on.
>
> 1. **The join target is a set-returning function, not a table.** `item` is
>    `jsonb_array_elements(events.payload -> 'items')`, which yields one row per array element.
>    Because its argument mentions `events.payload`, Postgres treats it as a `LATERAL` join
>    (it is allowed to leave out the keyword for functions in `FROM`): for each `events` row it
>    evaluates the function on *that* row's payload and pairs the row with each element it returns.
>    So it is not "all events x all items"; it is per-row unnesting.
> 2. **`ON true` is the join condition.** There is nothing to match on, the pairing is already done by
>    the lateral evaluation, so the condition is simply always true. Writing `.join(item, true())`
>    is SQLAlchemy's way of spelling `CROSS JOIN LATERAL`.
>
> Rows whose payload has no `items` key: `payload -> 'items'` is `NULL`, `jsonb_array_elements(NULL)`
> returns zero rows, and an inner join with zero rows on the right drops the event. That is why only the
> two purchases appear. Use `.outerjoin(item, true())` (`LEFT JOIN ... ON true`) to keep every event and
> get `NULL` for `value`.
>
> `.select_from(Event)` is required here only because no column in the `SELECT` list belongs to `events`
> (`sku`, `qty`, `price` all come from `item`), so SQLAlchemy would not know which table goes on the left
> of the `JOIN`.

In [87]:
# ... or pull the unnested rows and aggregate in pandas: same answer, pick whichever is handier.
items_df.assign(revenue=items_df.qty * items_df.price).groupby("sku")[
    ["qty", "revenue"]
].sum().sort_values("revenue", ascending=False)

,qty,revenue
sku,,
B-7,1,2500.50
A-1,2,2000.00
C-3,1,120.00


In [ ]:
# jsonb_to_recordset: straight into typed columns, no ->> and no casts
rec = (
    func.jsonb_to_recordset(Event.payload["items"])
    .table_valued(
        column("sku", Text), column("qty", Integer), column("price", Numeric(12, 2))
    )
    .render_derived(with_types=True)
)
await q(
    select(Event.id, rec.c.sku, rec.c.qty, rec.c.price)
    .join(rec, true())  # What does this mean? On what is join performed?
    .order_by(Event.id.desc(), rec.c.sku.asc())
)

SELECT events.id, anon_1.sku, anon_1.qty, anon_1.price 
FROM events JOIN jsonb_to_recordset(events.payload[%(payload_1)s]) AS anon_1(sku TEXT, qty INTEGER, price NUMERIC(12, 2)) ON true ORDER BY events.id DESC, anon_1.sku ASC
-- params: {'payload_1': 'items'}
2026-09-04 20:44:47,132 INFO sqlalchemy.engine.Engine SELECT events.id, anon_1.sku, anon_1.qty, anon_1.price 
FROM events JOIN jsonb_to_recordset(events.payload[%(payload_1)s::TEXT]) AS anon_1(sku TEXT, qty INTEGER, price NUMERIC(12, 2)) ON true ORDER BY events.id DESC, anon_1.sku ASC
2026-09-04 20:44:47,132 INFO sqlalchemy.engine.Engine [generated in 0.00040s] {'payload_1': 'items'}


,id,sku,qty,price
0,119,C-3,1,120.00
1,118,A-1,2,1000.00
2,118,B-7,1,2500.50


> **Q: `.join(rec, true())`, what does it mean, joined on what?**
>
> Same mechanism as in the `jsonb_array_elements` cell above: `rec` is the set-returning function
> `jsonb_to_recordset(events.payload -> 'items')`, Postgres evaluates it laterally for each `events`
> row, and `ON true` says "keep every pair". Compiled:
>
> ```sql
> FROM events JOIN jsonb_to_recordset(events.payload -> 'items')
>      AS anon_1(sku TEXT, qty INTEGER, price NUMERIC(12, 2)) ON true
> ```
>
> The difference from `jsonb_array_elements` is the output shape: instead of one `value jsonb` column
> that you then pick apart with `->>` and casts, `jsonb_to_recordset` produces real typed columns.
> Postgres cannot guess those types, which is why the alias *must* list them
> (`AS anon_1(sku TEXT, qty INTEGER, ...)`), and why the SQLAlchemy side needs
> `.render_derived(with_types=True)`. Keys that are missing in an object come back as `NULL` in that column.

In [92]:
await q(select(Event.payload["tags"].label("tags")))
# Why without label it shows as no column? Like absence of AS in SQL?

SELECT events.payload[%(payload_1)s] AS tags 
FROM events
-- params: {'payload_1': 'tags'}
2026-09-04 20:44:57,264 INFO sqlalchemy.engine.Engine SELECT events.payload[%(payload_1)s::TEXT] AS tags 
FROM events
2026-09-04 20:44:57,264 INFO sqlalchemy.engine.Engine [cached since 421.1s ago] {'payload_1': 'tags'}


,tags
0,"[returning, 2fa]"
1,[new]
2,[promo]
3,[]
4,[api]


> **Q: why does the column come back as `anon_1` without `.label()`? Like a missing `AS`?**
>
> Almost. Postgres always gives a result column *some* name: for a plain column it is the column name,
> for a function call it is the function name, and for an operator expression such as `payload -> 'tags'`
> it is the placeholder `?column?`.
>
> SQLAlchemy does not leave that to chance. Any expression that is not a bare column gets an automatic
> alias, so the SQL it sends is actually
>
> ```sql
> SELECT events.payload -> 'tags' AS anon_1 FROM events
> ```
>
> and `anon_1` is the key you see in the DataFrame. `Event.payload` alone keeps its name (`payload`)
> because it is a real column. So: raw column, no label needed; anything derived (`->`, `->>`, casts,
> `func.*`), add `.label("...")` if you want to refer to it by name in pandas, in `.group_by()`, or in
> `.order_by()`.

In [93]:
# jsonb_array_elements_text for arrays of scalars like tags [returning, 2fa]
tag = (
    func.jsonb_array_elements_text(Event.payload["tags"])
    .table_valued(column("value", Text))  # Don't fully get this
    .render_derived()  # And this
)
await q(
    select(tag.c.value.label("tag"), func.count().label("events"))
    .select_from(Event)
    .join(tag, true())
    .group_by(tag.c.value)
    .order_by(func.count().desc(), tag.c.value)
)

SELECT anon_1.value AS tag, count(*) AS events 
FROM events JOIN jsonb_array_elements_text(events.payload[%(payload_1)s]) AS anon_1(value) ON true GROUP BY anon_1.value ORDER BY count(*) DESC, anon_1.value
-- params: {'payload_1': 'tags'}
2026-09-04 20:45:02,415 INFO sqlalchemy.engine.Engine SELECT anon_1.value AS tag, count(*) AS events 
FROM events JOIN jsonb_array_elements_text(events.payload[%(payload_1)s::TEXT]) AS anon_1(value) ON true GROUP BY anon_1.value ORDER BY count(*) DESC, anon_1.value
2026-09-04 20:45:02,416 INFO sqlalchemy.engine.Engine [cached since 541s ago] {'payload_1': 'tags'}


,tag,events
0,2fa,1
1,api,1
2,new,1
3,promo,1
4,returning,1


> **Q: what do `.table_valued(column("value", Text))` and `.render_derived()` do?**
>
> `func.jsonb_array_elements_text(...)` on its own is a *scalar* expression: SQLAlchemy would happily
> put it in a `SELECT` list, but it has no idea that it can also sit in `FROM` and produce rows and
> columns.
>
> - **`.table_valued(...)`** turns it into a `FROM`-able object and declares the columns it produces.
>   Postgres's `jsonb_array_elements_text` returns a single column named `value` of type `text`, but
>   SQLAlchemy cannot look that up, so you tell it: `column("value", Text)`. The payoff is `tag.c.value`,
>   which you can then select, filter, and group by like any other column (`column()` here is the
>   lightweight, table-less column constructor from `sqlalchemy`).
> - **`.render_derived()`** controls how the alias is written. Without it SQLAlchemy emits
>   `... AS anon_1`; with it, `... AS anon_1(value)`, i.e. the alias also *names the columns*
>   (a "derived column list" in SQL terms). For `jsonb_array_elements_text` both work, because Postgres
>   already calls the column `value`. It becomes mandatory for `jsonb_to_recordset`, where the column
>   list (with types, `render_derived(with_types=True)`) is the only way Postgres learns what to
>   return. Using it everywhere keeps the pattern uniform.
>
> Side by side:
>
> ```sql
> -- .table_valued(column("value", Text))
> FROM events JOIN jsonb_array_elements_text(events.payload -> 'tags') AS anon_1 ON true
> -- .table_valued(column("value", Text)).render_derived()
> FROM events JOIN jsonb_array_elements_text(events.payload -> 'tags') AS anon_1(value) ON true
> ```

In [ ]:
# jsonb_path_query: set-returning JSONPath. The path must be typed JSONPATH.
m = (
    func.jsonb_path_query(
        Event.payload,
        literal(
            "$.items[*] ? (@.price > 500)", JSONPATH
        ),  # PartiQL like? Need an explainer.
    )
    .table_valued(column("value", JSONB))
    .render_derived()
)
await q(select(Event.id, m.c.value).join(m, true()).order_by(Event.id))

SELECT events.id, anon_1.value 
FROM events JOIN jsonb_path_query(events.payload, %(param_1)s) AS anon_1(value) ON true ORDER BY events.id
-- params: {'param_1': '$.items[*] ? (@.price > 500)'}
2026-09-04 20:45:47,133 INFO sqlalchemy.engine.Engine SELECT events.id, anon_1.value 
FROM events JOIN jsonb_path_query(events.payload, %(param_1)s) AS anon_1(value) ON true ORDER BY events.id
2026-09-04 20:45:47,133 INFO sqlalchemy.engine.Engine [cached since 435.7s ago] {'param_1': '$.items[*] ? (@.price > 500)'}


,id,value
0,123,"{'qty': 2, 'sku': 'A-1', 'price': 1000}"
1,123,"{'qty': 1, 'sku': 'B-7', 'price': 2500.5}"


> **Q: is JSONPath PartiQL-like? Explainer, please.**
>
> Not PartiQL. PartiQL is Amazon's SQL dialect for semi-structured data (DynamoDB, Redshift SUPER).
> What Postgres implements is the **SQL/JSON path language** from the SQL:2016 standard, available
> since Postgres 12, and it is a close cousin of the JSONPath used in JavaScript tooling and of `jq`.
> Think of it as "a small query language that runs *inside* one JSON document".
>
> Grammar you will actually use:
>
> | Syntax | Meaning |
> |---|---|
> | `$` | the document passed in (`payload`) |
> | `$.user.name` | key access |
> | `$.items[*]` | every element of an array; `[0]`, `[last]`, `[0 to 2]` also work |
> | `$.**` | any depth (recursive descent) |
> | `? (...)` | filter: keep only elements for which the predicate holds |
> | `@` | inside a filter, "the current element" |
> | `==  !=  <  >  <=  >=  &&  \|\|  !` | comparisons / boolean logic |
> | `like_regex "..."`, `starts with "..."`, `exists (...)` | more predicates |
> | `.type()`, `.size()`, `.double()`, `.datetime()` | item methods |
> | `$var` | variable, supplied through the `vars` jsonb argument |
>
> So `$.items[*] ? (@.price > 500)` reads: "each element of `items` whose `price` is greater than 500".
> Missing keys in lax mode (the default) just yield nothing instead of erroring; prefix the path with
> `strict` to make structural mismatches raise.
>
> The path functions and their operator shorthands:
>
> | Function | Operator | Returns |
> |---|---|---|
> | `jsonb_path_exists(doc, path)` | `doc @? path` | boolean: does the path match anything? (`Event.payload.path_exists(...)` above) |
> | `jsonb_path_match(doc, path)` | `doc @@ path` | boolean result of a single predicate, e.g. `$.amount > 100` |
> | `jsonb_path_query(doc, path)` | | **set** of matches, one row each (this cell) |
> | `jsonb_path_query_array(doc, path)` | | all matches packed into one jsonb array |
> | `jsonb_path_query_first(doc, path)` | | first match or `NULL` |
>
> `@?` and `@@` can use a `jsonb_path_ops` GIN index; the set-returning ones cannot.
>
> **Why `literal(..., JSONPATH)`?** The function signature is `(jsonb, jsonpath)`. SQLAlchemy sends the path
> as a bound parameter, and with asyncpg an untyped string is rendered as `$1::VARCHAR`, which makes
> Postgres look for `jsonb_path_query(jsonb, character varying)` and fail. Typing it as `JSONPATH`
> drops the cast and lets Postgres infer the `jsonpath` parameter from the function signature.

## 4. Building JSON from rows, and updating documents

In [ ]:
await q(
    select(
        Event.kind,
        # Create 'events' list of dicts as a custom collection of attributes
        func.jsonb_agg(
            func.jsonb_build_object(
                "id", Event.id, "user", Event.payload[("user", "name")].astext
            )
        ).label("events"),
    )
    .group_by(Event.kind)
    .order_by(Event.kind)
)

SELECT events.kind, jsonb_agg(jsonb_build_object(%(jsonb_build_object_1)s, events.id, %(jsonb_build_object_2)s, events.payload #>> %(payload_1)s)) AS events 
FROM events GROUP BY events.kind ORDER BY events.kind
-- params: {'jsonb_build_object_1': 'id', 'jsonb_build_object_2': 'user', 'payload_1': ('user', 'name')}
2026-09-04 20:46:26,238 INFO sqlalchemy.engine.Engine SELECT events.kind, jsonb_agg(jsonb_build_object(%(jsonb_build_object_1)s::VARCHAR, events.id, %(jsonb_build_object_2)s::VARCHAR, events.payload #>> %(payload_1)s)) AS events 
FROM events GROUP BY events.kind ORDER BY events.kind
2026-09-04 20:46:26,239 INFO sqlalchemy.engine.Engine [cached since 35.04s ago] {'jsonb_build_object_1': 'id', 'jsonb_build_object_2': 'user', 'payload_1': '{user, name}'}


,kind,events
0,error,"[{'id': 125, 'user': None}]"
1,login,"[{'id': 122, 'user': 'Bola'}, {'id': 121, 'user': 'Ada'}]"
2,purchase,"[{'id': 124, 'user': 'Chidi'}, {'id': 123, 'user': 'Ada'}]"


In [102]:
# || merges top-level keys (right side wins; nested objects are replaced, not merged)
await q(
    update(Event)  # What is the SQL equivalent?
    .where(Event.kind == "error")
    .values(payload=Event.payload.concat({"reviewed": True}))
    .returning(Event.id, Event.payload)  # What is returning?
)

UPDATE events SET payload=(events.payload || %(payload_1)s::JSONB) WHERE events.kind = %(kind_1)s RETURNING events.id, events.payload
-- params: {'payload_1': {'reviewed': True}, 'kind_1': 'error'}
2026-09-04 21:10:41,954 INFO sqlalchemy.engine.Engine UPDATE events SET payload=(events.payload || %(payload_1)s::JSONB) WHERE events.kind = %(kind_1)s::VARCHAR RETURNING events.id, events.payload
2026-09-04 21:10:41,956 INFO sqlalchemy.engine.Engine [cached since 893.4s ago] {'payload_1': Jsonb({'reviewed': True}), 'kind_1': 'error'}


,id,payload
0,125,"{'code': 500, 'tags': ['api'], 'message': 'upstream timeout', 'reviewed': True}"


> **Q: what SQL does `update(Event)` produce, and what is `RETURNING`?**
>
> The cell compiles to
>
> ```sql
> UPDATE events
> SET payload = (events.payload || '{"reviewed": true}'::jsonb)
> WHERE events.kind = 'error'
> RETURNING events.id, events.payload
> ```
>
> `update(Event)` is `UPDATE events`; `.values(...)` is the `SET` list; `.where(...)` is the `WHERE`.
> `.concat({...})` is the jsonb `||` operator, and SQLAlchemy serialises the dict and casts it to `jsonb`
> because the column type is `JSONB`.
>
> **`RETURNING`** is a Postgres extension. By default `UPDATE`, `INSERT` and `DELETE` only report how many
> rows were affected. `RETURNING <columns>` makes the statement also send back those columns from every
> affected row, *after* the change, in the same round trip. So you get "update and show me the result"
> without a second `SELECT`, and without a race against other writers. That is why `q()` can turn an
> `update()` into a DataFrame: the result set looks exactly like a `SELECT`. Any expression is allowed
> there, e.g. `Event.payload["user"].label("user")` two cells down.
>
> Note that `q()` executes but does not commit; the change is visible in this session's transaction and
> becomes permanent at the `session.commit()` in the ORM cell below (or is discarded on rollback).

In [105]:
# jsonb_set(target, text[] path, jsonb value): typing the literal as JSONB JSON-encodes it
await q(
    update(Event)
    .where(Event.payload.contains({"user": {"id": 1}}))
    .values(
        payload=func.jsonb_set(
            Event.payload, array(["user", "plan"]), literal("enterprise", JSONB)
        )
    )
    .returning(Event.id, Event.payload["user"].label("user"))
)

UPDATE events SET payload=jsonb_set(events.payload, ARRAY[%(param_1)s, %(param_2)s], %(param_3)s::JSONB) WHERE events.payload @> %(payload_1)s::JSONB RETURNING events.id, events.payload[%(payload_2)s] AS "user"
-- params: {'param_1': 'user', 'param_2': 'plan', 'param_3': 'enterprise', 'payload_1': {'user': {'id': 1}}, 'payload_2': 'user'}
2026-09-04 21:10:51,562 INFO sqlalchemy.engine.Engine UPDATE events SET payload=jsonb_set(events.payload, ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR], %(param_3)s::JSONB) WHERE events.payload @> %(payload_1)s::JSONB RETURNING events.id, events.payload[%(payload_2)s::TEXT] AS "user"
2026-09-04 21:10:51,563 INFO sqlalchemy.engine.Engine [cached since 917.3s ago] {'param_1': 'user', 'param_2': 'plan', 'param_3': Jsonb('enterprise'), 'payload_1': Jsonb({'user': {'id': 1}}), 'payload_2': 'user'}


,id,user
0,121,"{'id': 1, 'name': 'Ada', 'plan': 'enterprise', 'country': 'NG'}"
1,123,"{'id': 1, 'name': 'Ada', 'plan': 'enterprise', 'country': 'NG'}"


In [ ]:
# - drops a top-level key (type the key as Text!), #- drops by path
await q(
    update(Event)
    .where(Event.kind == "error")
    .values(payload=Event.payload.op("-")(literal("tags", Text)))
    .returning(Event.id, Event.payload)
)
await q(
    update(Event)
    .where(Event.payload.contains({"user": {"id": 1}}))
    .values(payload=Event.payload.delete_path(["user", "plan"]))
    .returning(Event.id, Event.payload["user"].label("user"))
)

# Note: user id 1 (Ada) is on two rows, a login and a purchase, so the second
# update changes and returns both. See the note below.

UPDATE events SET payload=(events.payload - %(param_1)s) WHERE events.kind = %(kind_1)s RETURNING events.id, events.payload
-- params: {'param_1': 'tags', 'kind_1': 'error'}
2026-09-04 21:10:53,240 INFO sqlalchemy.engine.Engine UPDATE events SET payload=(events.payload - %(param_1)s::VARCHAR) WHERE events.kind = %(kind_1)s::VARCHAR RETURNING events.id, events.payload
2026-09-04 21:10:53,240 INFO sqlalchemy.engine.Engine [cached since 7.882s ago] {'param_1': 'tags', 'kind_1': 'error'}
UPDATE events SET payload=(events.payload #- CAST(ARRAY[%(param_1)s, %(param_2)s] AS TEXT[])) WHERE events.payload @> %(payload_1)s::JSONB RETURNING events.id, events.payload[%(payload_2)s] AS "user"
-- params: {'param_1': 'user', 'param_2': 'plan', 'payload_1': {'user': {'id': 1}}, 'payload_2': 'user'}
2026-09-04 21:10:53,243 INFO sqlalchemy.engine.Engine UPDATE events SET payload=(events.payload #- CAST(ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR] AS TEXT[])) WHERE events.payload @> %(payload_1)s::J

,id,user
0,121,"{'id': 1, 'name': 'Ada', 'country': 'NG'}"
1,123,"{'id': 1, 'name': 'Ada', 'country': 'NG'}"


> **Which rows do these two updates touch?**
>
> - The first (`payload - 'tags'`) filters on `kind = 'error'`: only the error row, whose `tags` array
>   (`["api"]`) disappears.
> - The second (`#- '{user,plan}'`) filters on `payload @> '{"user": {"id": 1}}'`. User 1 (Ada) appears in
>   **two** rows of different kinds: her `login` and her `purchase`. Both are updated and both come back
>   in `RETURNING`. It undoes the `jsonb_set` two cells up, which put `"plan": "enterprise"` on the same
>   two rows. The error row has no `user` key at all, so `contains` is false for it and it is untouched.
>
> That is the general point of filtering on document content rather than on `kind`: the same person can
> show up under several event types, and a `@>` filter catches all of them.

In [107]:
# ORM style: mutating the dict in place is invisible to SQLAlchemy; reassign the attribute.
ev = await session.scalar(select(Event).where(Event.kind == "error"))
ev.payload = {**ev.payload, "retries": 3}
await session.commit()
await q(select(Event.id, Event.payload).where(Event.kind == "error"), show=False)

2026-09-04 21:11:57,084 INFO sqlalchemy.engine.Engine SELECT events.id, events.kind, events.payload, events.created_at 
FROM events 
WHERE events.kind = %(kind_1)s::VARCHAR
2026-09-04 21:11:57,085 INFO sqlalchemy.engine.Engine [generated in 0.00129s] {'kind_1': 'error'}
2026-09-04 21:11:57,091 INFO sqlalchemy.engine.Engine UPDATE events SET payload=%(payload)s::JSONB WHERE events.id = %(events_id)s::INTEGER
2026-09-04 21:11:57,091 INFO sqlalchemy.engine.Engine [generated in 0.00046s] {'payload': Jsonb({'code': 500, 'message': 'upstream  ... (76 chars)), 'events_id': 125}
2026-09-04 21:11:57,092 INFO sqlalchemy.engine.Engine COMMIT
2026-09-04 21:11:57,095 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 21:11:57,096 INFO sqlalchemy.engine.Engine SELECT events.id, events.payload 
FROM events 
WHERE events.kind = %(kind_1)s::VARCHAR
2026-09-04 21:11:57,097 INFO sqlalchemy.engine.Engine [generated in 0.00071s] {'kind_1': 'error'}


,id,payload
0,125,"{'code': 500, 'message': 'upstream timeout', 'retries': 3, 'reviewed': True}"


## 5. Exploring with pandas

For ad-hoc exploration it is often quickest to pull the raw documents and flatten them with
`pd.json_normalize`. Nested keys become dotted columns; arrays stay as lists.

In [108]:
raw = await q(
    select(Event.id, Event.kind, Event.payload).order_by(Event.id), show=False
)
flat = pd.concat(
    [raw[["id", "kind"]], pd.json_normalize(raw["payload"].tolist())], axis=1
)
flat

2026-09-04 21:12:23,864 INFO sqlalchemy.engine.Engine SELECT events.id, events.kind, events.payload 
FROM events ORDER BY events.id
2026-09-04 21:12:23,865 INFO sqlalchemy.engine.Engine [cached since 2531s ago] {}


,id,kind,tags,device,user.id,user.name,user.country,items,amount,currency,refunded,code,message,retries,reviewed
0,121,login,"[returning, 2fa]",web,1.0,Ada,NG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,122,login,[new],mobile,2.0,Bola,GH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,123,purchase,[promo],NaN,1.0,Ada,NG,"[{'qty': 2, 'sku': 'A-1', 'price': 1000}, {'qty': 1, 'sku': 'B-7', 'price': 2500.5}]",4500.5,NGN,NaN,NaN,NaN,NaN,NaN
3,124,purchase,[],NaN,3.0,Chidi,NG,"[{'qty': 1, 'sku': 'C-3', 'price': 120}]",120.0,NGN,True,NaN,NaN,NaN,NaN
4,125,error,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,upstream timeout,3.0,True


In [109]:
# explode the items array into rows, then normalise the item objects
exploded = flat[["id", "items"]].explode("items").dropna()
pd.concat(
    [
        exploded[["id"]].reset_index(drop=True),
        pd.json_normalize(exploded["items"].tolist()),
    ],
    axis=1,
)

,id,qty,sku,price
0,123,2,A-1,1000.0
1,123,1,B-7,2500.5
2,124,1,C-3,120.0


In [110]:
# Which keys does each document have? Handy on messy real-world data.
flat.notna().groupby(flat["kind"]).sum().T

kind,error,login,purchase
id,1,2,2
kind,1,2,2
tags,0,2,2
device,0,2,0
user.id,0,2,2
user.name,0,2,2
user.country,0,2,2
items,0,0,2
amount,0,0,2
currency,0,0,2


## 6. Your turn

Some things to try:

- Add a row with a shape nobody expects (e.g. `"items"` as an object instead of a list) and see
  what breaks in section 3 and how `jsonb_typeof` can guard against it.
- Change `.contains({"tags": ["promo"]})` to `.has_key("promo")` and explain why it returns nothing.
- Build a GIN index (`Index("ix", Event.payload, postgresql_using="gin", postgresql_ops={"payload": "jsonb_path_ops"})`)
  and compare `EXPLAIN` output with `SET enable_seqscan = off`.

`await reset_events(session)` restores the five sample rows at any time.

In [111]:
await reset_events(session)
await session.close()

2026-09-04 21:12:49,926 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 21:12:49,926 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR
2026-09-04 21:12:49,927 INFO sqlalchemy.engine.Engine [cached since 2625s ago] {'table_name': 'events', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-09-04 21:12:49,931 INFO sqlalchemy.engine.Engine COMMIT
2026-09-04 21:12:49,934 INFO sqlalchemy.engine.Engine DELETE FROM events
2026-09-04 21:12:49,934 INFO sqlalc